In [2]:
!pip -q install pandas numpy scipy

from google.colab import files
from pathlib import Path
uploaded = files.upload()
inputs = [Path(filename) for filename in uploaded]
if not inputs:
    raise ValueError('Upload at least one model results ZIP or prediction CSV')
print('Uploaded:', [x.name for x in inputs])

Saving early_fusion_results (1).zip to early_fusion_results (1).zip
Uploaded: ['early_fusion_results (1).zip']


In [3]:

from __future__ import annotations
import argparse
import os
import re
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

FUSIONS = ('early', 'intermediate', 'late', 'graph')
SEXES = ('Female', 'Male')
BASE_COLS = ('sample_id', 'cell_line_id', 'drug_id', 'sex', 'response',
             'prediction', 'split_family', 'fusion', 'sex_aware', 'proteomics')
EF_MAP = {'EF1': (False, False), 'EF2': (True, False),
          'EF3': (False, True), 'EF4': (True, True)}


def parse_bool(value, *, field):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        raise ValueError(f'Missing {field}: provide True/False')
    s = str(value).strip().lower()
    if s in ('true', '1', 'yes', 'y', 'with', 'included'):
        return True
    if s in ('false', '0', 'no', 'n', 'without', 'excluded'):
        return False
    raise ValueError(f'Invalid {field} value: {value!r}; use True/False')


def open_inputs(paths, workdir):
    """Support ordinary CSVs, directory trees and ZIPs of results."""
    workdir.mkdir(exist_ok=True, parents=True)
    candidates = []
    for n, item in enumerate(paths):
        p = Path(item).expanduser()
        if not p.exists():
            raise FileNotFoundError(p)
        if p.is_dir():
            candidates.extend(sorted(p.rglob('*.csv')))
        elif p.suffix.lower() == '.zip':
            extracted = workdir / f'archive_{n:02d}'
            extracted.mkdir(exist_ok=True)
            with zipfile.ZipFile(p) as z:
                for info in z.infolist():
                    target = (extracted / info.filename).resolve()
                    if not target.is_relative_to(extracted.resolve()):
                        raise ValueError(f'Unsafe ZIP member: {info.filename}')
                z.extractall(extracted)
            candidates.extend(sorted(extracted.rglob('*.csv')))
        elif p.suffix.lower() == '.csv':
            candidates.append(p)
        else:
            raise ValueError(f'Expected a CSV, ZIP or directory, got: {p}')
    # Don't accidentally process benchmark or labels as model predictions
    result = []
    seen = set()
    for p in candidates:
        if 'prediction' not in p.name.lower():
            continue
        if p.resolve() not in seen:
            seen.add(p.resolve())
            result.append(p)
    if not result:
        raise ValueError('No *prediction*.csv found. Export one CSV per model or a combined predictions CSV.')
    return result


def normalize_predictions(path):
    df = pd.read_csv(path, low_memory=False)
    if df.empty:
        raise ValueError(f'{path}: empty input')
    rename = {
        'y_true': 'response', 'y_pred': 'prediction', 'true': 'response',
        'pred': 'prediction', 'split_type': 'split_family',
        'cell_id': 'cell_line_id',
    }
    for old, new in rename.items():
        if old in df and new not in df:
            df = df.rename(columns={old: new})
    # Original early-fusion notebook labels versions EF1 through EF4.
    if 'model' in df and 'fusion' in df and df.fusion.astype(str).str.lower().eq('early').all():
        models = df.model.astype(str).str.upper().str.strip()
        unknown = set(models.unique()) - set(EF_MAP)
        if unknown and ('sex_aware' not in df or 'proteomics' not in df):
            raise ValueError(f'{path.name}: unknown EF model codes {unknown}. '
                             'Provide explicit sex_aware and proteomics columns.')
        if not unknown:
            exp_prot = models.map(lambda m: EF_MAP[m][0])
            exp_sex = models.map(lambda m: EF_MAP[m][1])
            if 'proteomics' not in df:
                df['proteomics'] = exp_prot
            if 'sex_aware' not in df:
                df['sex_aware'] = exp_sex
    if 'sample_id' not in df and {'cell_line_id', 'drug_id'}.issubset(df.columns):
        df['sample_id'] = df.cell_line_id.astype(str) + '__' + df.drug_id.astype(str)
    if 'split' not in df:
        raise ValueError(f'{path.name}: missing split (only frozen TEST predictions qualify)')
    split = df.split.astype(str).str.lower().str.strip()
    if not split.eq('test').any():
        raise ValueError(f'{path.name}: no split=test rows')
    df = df.loc[split.eq('test')].copy()
    missing = sorted(set(BASE_COLS) - set(df.columns))
    if missing:
        raise ValueError(f'{path.name}: missing {missing}. '
                         'For other fusion architectures, include explicit fusion, sex_aware, proteomics.')
    df['fusion'] = df.fusion.astype(str).str.lower().str.strip().replace({
        'early fusion':'early', 'intermediate fusion':'intermediate',
        'late fusion':'late', 'graph fusion':'graph'
    })
    if not set(df.fusion).issubset(FUSIONS):
        raise ValueError(f'{path.name}: unexpected fusion values {sorted(set(df.fusion)-set(FUSIONS))}')
    df['sex'] = df.sex.astype(str).str.strip().str.title()
    if not set(df.sex).issubset(SEXES):
        raise ValueError(f'{path.name}: unexpected sex labels {sorted(set(df.sex)-set(SEXES))}. '
                         'Use only QC-usable Female/Male lines.')
    for field in ('sex_aware','proteomics'):
        df[field] = [parse_bool(v, field=field) for v in df[field]]
    for field in ('sample_id','cell_line_id','drug_id','split_family'):
        df[field] = df[field].astype(str).str.strip()
    for field in ('response', 'prediction'):
        df[field] = pd.to_numeric(df[field], errors='coerce')
    if not np.isfinite(df[['response','prediction']].to_numpy(dtype=float)).all():
        raise ValueError(f'{path.name}: NaN or infinite response/prediction values. Fix model outputs first.')
    df['source_file'] = path.name
    df = df[list(BASE_COLS) + ['source_file']]
    if df.duplicated(['split_family','proteomics','fusion','sex_aware','sample_id']).any():
        raise ValueError(f'{path.name}: duplicate per-model test predictions; fix before aggregating')
    return df


def checked_concat(parts):
    df = pd.concat(parts, ignore_index=True)
    dupes = df.duplicated(['split_family','proteomics','fusion','sex_aware','sample_id'])
    if dupes.any():
        example = df.loc[dupes, ['fusion','sex_aware','proteomics','source_file']].head().to_dict('records')
        raise ValueError(f'Duplicate model predictions across uploaded files: {example}. '
                         'Upload each results file only once.')
    return df


def metric_values(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    n = len(y)
    if n == 0:
        return dict(n=0, RMSE=np.nan, MAE=np.nan, Pearson=np.nan, Spearman=np.nan)
    rmse = float(np.sqrt(np.mean((y-p)**2)))
    mae = float(np.mean(np.abs(y-p)))
    pearson = float(pearsonr(y, p).statistic) if n >= 3 and np.std(y)>0 and np.std(p)>0 else np.nan
    spearman = float(spearmanr(y, p).statistic) if n >= 3 and len(set(y))>1 and len(set(p))>1 else np.nan
    return dict(n=n, RMSE=rmse, MAE=mae, Pearson=pearson, Spearman=spearman)


def build_benchmark(df):
    rows = []
    cohort_rows = []
    matched_predictions = []
    for (family, prot), g in df.groupby(['split_family','proteomics'], sort=True):
        variants = list(g.groupby(['fusion','sex_aware'], sort=True))
        idsets = [set(x.sample_id) for _, x in variants]
        shared = set.intersection(*idsets)
        if not shared:
            raise ValueError(f'{family}, proteomics={prot}: model test sample IDs have zero intersection')
        # Compare truth and sex metadata on ALL overlapping instances across models.
        common = g[g.sample_id.isin(shared)].copy()
        checks = common.groupby('sample_id', sort=False).agg(
            n_responses=('response','nunique'),
            n_sex=('sex','nunique'),
            n_cell=('cell_line_id','nunique'),
            n_drug=('drug_id','nunique')
        )
        if (checks[['n_responses','n_sex','n_cell','n_drug']] != 1).any().any():
            raise ValueError(f'{family}, proteomics={prot}: response/sex/IDs differ '
                             'for the same test sample across models. Check endpoint, ID harmonization and QC.')
        if len(shared) < min(len(s) for s in idsets):
            warnings.warn(f'{family}, proteomics={prot}: test-set intersection contains '
                          f'{len(shared)} observations; some samples excluded for cross-model comparability.')
        for (fusion, aware), mdf in variants:
            mdf = mdf[mdf.sample_id.isin(shared)].sort_values('sample_id')
            if len(mdf) != len(shared):
                raise AssertionError('Missing samples despite intersection')
            matched_predictions.append(mdf)
            fm = metric_values(mdf.loc[mdf.sex=='Female','response'],
                               mdf.loc[mdf.sex=='Female','prediction'])
            mm = metric_values(mdf.loc[mdf.sex=='Male','response'],
                               mdf.loc[mdf.sex=='Male','prediction'])
            allm = metric_values(mdf.response, mdf.prediction)
            for sex_name, metric in [('Female',fm),('Male',mm)]:
                rows.append(dict(split_family=family, proteomics=bool(prot),
                                 fusion=fusion, sex_aware=bool(aware), group=sex_name,
                                 **metric))
            gap = {metric: abs(fm[metric] - mm[metric])
                   for metric in ('RMSE','MAE','Pearson','Spearman')}
            rows.append(dict(split_family=family, proteomics=bool(prot),
                             fusion=fusion, sex_aware=bool(aware), group='Gap (absolute)',
                             n=np.nan, **gap))
            cohort_rows.append(dict(split_family=family, proteomics=bool(prot),
                                    fusion=fusion, sex_aware=bool(aware),
                                    original_n=len(g[(g.fusion==fusion)&(g.sex_aware==aware)]),
                                    common_test_n=len(shared),
                                    original_female_n=int((g[(g.fusion==fusion)&(g.sex_aware==aware)].sex=='Female').sum()),
                                    common_female_n=fm['n'], common_male_n=mm['n']))
    long = pd.DataFrame(rows)
    cohort = pd.DataFrame(cohort_rows)
    matched = pd.concat(matched_predictions, ignore_index=True)
    # One wide row for each model condition.
    wide = []
    for (family,prot,fusion,aware), sub in long.groupby(['split_family','proteomics','fusion','sex_aware'], sort=True):
        fm = sub.set_index('group')
        row = {'split_family':family, 'proteomics':prot,'fusion':fusion,'sex_aware':aware,
               'n_female':int(fm.loc['Female','n']), 'n_male':int(fm.loc['Male','n']),
               'n_total':int(fm.loc['Female','n']+fm.loc['Male','n'])}
        for metric in ('RMSE','MAE','Pearson','Spearman'):
            row[f'Female_{metric}']=fm.loc['Female',metric]
            row[f'Male_{metric}']=fm.loc['Male',metric]
            row[f'Gap_abs_{metric}']=fm.loc['Gap (absolute)',metric]
        row['Gap_signed_RMSE_F_minus_M'] = row['Female_RMSE'] - row['Male_RMSE']
        wide.append(row)
    return long, pd.DataFrame(wide), cohort, matched


def template_24row(long):
    """24 slots per (split-family, proteomics), with PENDING where not available."""
    expected = []
    for family, prot in sorted(set(zip(long.split_family,long.proteomics))):
        for fusion in FUSIONS:
            for aware in (False, True):
                for group in ('Female','Male','Gap (absolute)'):
                    expected.append(dict(split_family=family, proteomics=bool(prot),
                                         fusion=fusion, sex_aware=aware, group=group))
    exp = pd.DataFrame(expected)
    merged = exp.merge(long, on=['split_family','proteomics','fusion','sex_aware','group'],
                       how='left', validate='one_to_one', indicator=True)
    merged.insert(5, 'status', np.where(merged['_merge']=='both','MEASURED','PENDING'))
    return merged.drop(columns=['_merge'])


def coverage(long):
    combos = long[['split_family','proteomics','fusion','sex_aware']].drop_duplicates()
    out = []
    for family, prot in sorted(set(zip(long.split_family,long.proteomics))):
        have = set(zip(combos.loc[(combos.split_family==family)&(combos.proteomics==prot),'fusion'],
                       combos.loc[(combos.split_family==family)&(combos.proteomics==prot),'sex_aware']))
        missing = [(fusion, aware) for fusion in FUSIONS for aware in (False,True)
                   if (fusion,aware) not in have]
        out.append(dict(split_family=family, proteomics=bool(prot),
                        models_received=len(have), models_required=8,
                        complete=(len(missing)==0),
                        missing='; '.join(f'{fu} sex_aware={sa}' for fu,sa in missing)))
    return pd.DataFrame(out)


def run(inputs, output_dir, scratch=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    scratch = Path(scratch) if scratch else output_dir / '_extracted_inputs'
    inputfiles = open_inputs(inputs,scratch)
    print('Found',len(inputfiles),'prediction CSV file(s):')
    for p in inputfiles: print('  ',p)
    data = checked_concat([normalize_predictions(p) for p in inputfiles])
    long, wide, cohort, matched = build_benchmark(data)
    slot_table = template_24row(long)
    coverage_table = coverage(long)
    outputs = {
        'master_benchmark_long.csv':long,
        'master_benchmark_wide.csv':wide,
        'master_benchmark_24row_template.csv':slot_table,
        'model_coverage.csv':coverage_table,
        'matched_test_cohorts.csv':cohort,
        'matched_test_predictions.csv':matched,
    }
    for name, table in outputs.items():
        table.to_csv(output_dir/name,index=False)
        print(f'Saved {name}: {len(table)} rows')
    print('\nCoverage (per held-out split / proteomics condition):')
    print(coverage_table.to_string(index=False))
    print('\nMaster benchmark (available models only):')
    summary_cols=['split_family','proteomics','fusion','sex_aware',
                  'Female_RMSE','Male_RMSE','Gap_abs_RMSE']
    print(wide[summary_cols].round(4).to_string(index=False))
    return outputs

In [4]:
# 3. Evaluate all uploaded test predictions and download outputs.
import shutil
output_dir = Path('/content/tnam_master_benchmark_results')
result = run(inputs, output_dir)
zip_file = shutil.make_archive('/content/TNAM_Master_Benchmark_Results', 'zip', root_dir=output_dir,
                               base_dir='.')
print('Download:', zip_file)
files.download(zip_file)

Found 1 prediction CSV file(s):
   /content/tnam_master_benchmark_results/_extracted_inputs/archive_00/early_fusion_test_predictions.csv
Saved master_benchmark_long.csv: 12 rows
Saved master_benchmark_wide.csv: 4 rows
Saved master_benchmark_24row_template.csv: 48 rows
Saved model_coverage.csv: 2 rows
Saved matched_test_cohorts.csv: 4 rows
Saved matched_test_predictions.csv: 71584 rows

Coverage (per held-out split / proteomics condition):
     split_family  proteomics  models_received  models_required  complete                                                                                                                                           missing
unseen_cell_lines       False                2                8     False intermediate sex_aware=False; intermediate sex_aware=True; late sex_aware=False; late sex_aware=True; graph sex_aware=False; graph sex_aware=True
unseen_cell_lines        True                2                8     False intermediate sex_aware=False; intermediate 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>